# Train a model
In this example notebook, a model is trained for a location with id '287'. The data for this location can be found in the 'data' folder. 
First, the prediction job will be defined, which contains the properties of the training and prediction. For example the time horizon, machine learning model and location of the forecast are defined in the prediction job. 
Thereafter, the model can be trained using the input data and prediction job by the ```train_model_pipeline()```. 

In [2]:

import sys

sys.path.append("/Users/mfavit/forecasting-engine/") 

import pandas as pd
from IPython.display import IFrame
from openstef.pipeline.train_model import train_model_pipeline
from openstef.pipeline.create_forecast import create_forecast_pipeline
from openstef.data_classes.prediction_job import PredictionJobDataClass

/Users/mfavit/forecasting-engine/virtualenv_forecasting-engine/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Prepare for training
Before a model can be trained, the specifications and data need to be defined. The specification of the model are defined in the prediction job (pj), where for example the machine learning model, latitude, longtide and forecast horizon are specified. Furthermore, the data has to be retrieved from the csv file containing both load, weather and energy market data. 

In [3]:
# Define properties of training/prediction. We call this a 'prediction_job'
pj = dict(id=287,
        model='xgb', 
        quantiles=[10,30,50,70,90],
        forecast_type="demand",
        lat=52.0,
        lon=5.0,
        horizon_minutes=47*60,
        resolution_minutes=15,
        name="Example",          
        hyper_params={}, 
        feature_names=None, 
        default_modelspecs=None,
        save_train_forecasts=True,
       )
pj=PredictionJobDataClass(**pj)

# Load input data
input_data = pd.read_csv('data/get_model_input_pid_287.csv', index_col='index', parse_dates=True)

# Split in training and forecasting data. Everything except the last 20 rows will be used for training
train_data = input_data.iloc[:-200,:] # everything except last 200 rows (~ 48 hours)

In [4]:
# Print the train data. 
# For every timestamp, bot the load as well as feature data is available. 
display(train_data.head())

,load,APX,clouds,radiation,temp,winddeg,windspeed,windspeed_100m,pressure,humidity,...,sjv_E1A,sjv_E1B,sjv_E1C,sjv_E2A,sjv_E2B,sjv_E3A,sjv_E3B,sjv_E3C,sjv_E3D,sjv_E4A
index,,,,,,,,,,,,,,,,,,,,,
2020-10-02 09:45:00+00:00,3.310000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-10-02 10:00:00+00:00,2.620000,34.0,99.758911,1.552899e+06,16.449036,154.711456,3.527778,9.349441,99453.476562,0.686240,...,0.000031,0.000030,0.000029,0.000033,0.000032,0.000061,0.000048,0.000048,0.000031,0.0
2020-10-02 10:15:00+00:00,0.796667,34.0,99.819193,1.575618e+06,16.400948,157.491554,3.557639,9.232026,99416.363281,0.683780,...,0.000032,0.000030,0.000029,0.000033,0.000032,0.000060,0.000048,0.000048,0.000031,0.0
2020-10-02 10:30:00+00:00,0.300000,34.0,99.879475,1.598338e+06,16.352859,160.271652,3.587500,9.114612,99379.250000,0.681319,...,0.000032,0.000031,0.000029,0.000033,0.000031,0.000058,0.000048,0.000048,0.000031,0.0
2020-10-02 10:45:00+00:00,1.773333,34.0,99.939756,1.594736e+06,16.304771,163.051750,3.617361,8.997197,99342.136719,0.678859,...,0.000032,0.000030,0.000029,0.000032,0.000031,0.000057,0.000048,0.000048,0.000031,0.0


## Train a model
Train the model by using the high-level pipeline ```train_model_pipeline```. Store the model and reports on training proces in ./mlflow_artifacts and ./mlflow_trained_models by setting artifact_folder and mlflow_tracking_uri to this respective path. 

In [5]:
train, val, test=train_model_pipeline(
    pj,
    train_data,
    check_old_model_age=False,
    mlflow_tracking_uri="./mlflow_trained_models",
    artifact_folder="./mlflow_artifacts",
    )

2025/04/22 20:49:05 WARNING mlflow.tracking.fluent: Cannot retrieve experiment by name 287


2025-04-22 20:49:05 [warning  ] No old model found, training new model pid=287
2025-04-22 20:49:05 [info     ] Found 2 values of constant load (repeated values), converted to NaN value. cleansing_step=repeated_values frac_values=0.00017666283897182228 num_values=2 pj_id=287
2025-04-22 20:49:05 [info     ] Removed 2 NaN values           num_removed_values=2
[0]	validation_0-rmse:1.24582	validation_1-rmse:1.22236
[1]	validation_0-rmse:0.98600	validation_1-rmse:1.00451
[2]	validation_0-rmse:0.81787	validation_1-rmse:0.87191
[3]	validation_0-rmse:0.69701	validation_1-rmse:0.77576
[4]	validation_0-rmse:0.61416	validation_1-rmse:0.72240
[5]	validation_0-rmse:0.55203	validation_1-rmse:0.68648
[6]	validation_0-rmse:0.50657	validation_1-rmse:0.67186
[7]	validation_0-rmse:0.47056	validation_1-rmse:0.65842
[8]	validation_0-rmse:0.44299	validation_1-rmse:0.65054
[9]	validation_0-rmse:0.42236	validation_1-rmse:0.65162
[10]	validation_0-rmse:0.40114	validation_1-rmse:0.64741
[11]	validation_0-rmse:0

2025/04/22 20:49:12 INFO mlflow.tracking.fluent: Experiment with name '287' does not exist. Creating a new experiment.


2025-04-22 20:49:12 [info     ] No previous model found in MLflow experiment_name=287


Exception: './mlflow_trained_models' does not exist.

Now, you can find the trained model in ./mlflow_trained_models, along with reports on the training process. Below the Predictor0.25 and Predictor47.0 plots are shown, as well as the weight plot. The predictor plots show the prediction of the train, test and validation data. The weight plot shows the importance and weight of every feature.

In [13]:
# Inspect local files
display(IFrame('./mlflow_artifacts/{}/Predictor0.25.html'.format(pj['id']), width=900, height=400))
display(IFrame('./mlflow_artifacts/{}/Predictor47.0.html'.format(pj['id']), width=800, height=400))
display(IFrame('./mlflow_artifacts/{}/weight_plot.html'.format(pj['id']), width=800, height=400))


## Visual Studio Code has difficulties with displaying htmls. If you are working with VSC and are not able to inspect the plots, uncomment the code below
## to open the plots in your browser.
import webbrowser
webbrowser.open(r'.\mlflow_artifacts\{}\Predictor0.25.html'.format(pj['id']))
webbrowser.open(r'.\mlflow_artifacts\{}\Predictor47.0.html'.format(pj['id']))
webbrowser.open(r'.\mlflow_artifacts\{}\weight_plot.html'.format(pj['id']))

17:18: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)
119:120: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)
17:18: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)
119:120: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)
17:18: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)
119:120: syntax error: Expected “"” but found unknown token. (-2741)
118:119: syntax error: Expected “"” but found unknown token. (-2741)


False